# Dónde buscar arena de fractura

Vaca Muerta bombea más de cinco millones de toneladas de arena por año y las
trae en camión desde Entre Ríos, a 1.400 km. La arena cercana de Río Negro
quedó descartada por calidad, y Neuquén explora la suya: 104 pedidos de cateo
en tres zonas. Un geólogo que sale a buscar arena de fractura no busca "arena":
busca arena cuarzosa, redonda y resistente, y eso lo dan sobre todo los
depósitos eólicos, después los fluviales, y algunas areniscas poco cementadas.

Este notebook hace lo que hace ese geólogo antes de salir al campo, con datos
abiertos: lee el mapa geológico 1:250.000 de SEGEMAR, clasifica cada polígono
según su génesis, litología y edad, lo pondera por distancia al consumo, a las
rutas y a los depósitos conocidos, y devuelve un mapa de blancos. Después
controla el resultado contra los lugares donde la industria ya dijo, en
público, que hay arena buena o mala.

Lo que el mapa **no** dice: si la arena pasa la norma. Esfericidad, redondez,
rotura y arcillas se miden en laboratorio, y no hay un solo ensayo argentino
publicado como dato.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, time
sys.path.insert(0, "../src")
import geopandas as gpd, pandas as pd, numpy as np
from shapely.geometry import box
from IPython.display import IFrame, display

from alab import DATA_RAW, DATA_PROC, DOCS, FIGURES, BBOX
from alab.segemar import load_layer
from alab.targets import KW, classify_frame, normalize
from alab.score import WEIGHTS, prospectivity, top_targets
from alab.validate import known_points, score_at, summary
from alab.demand import read_fracture, read_well_coords, demand_by_area, demand_centroid
from alab.maps import interactive_map, static_map, class_bar, route_figure
from alab.route import sand_route, waypoints_frame, distance_to_route

pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
DATA_PROC.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

## 1. El mapa geológico y las reglas

La ventana cubre Neuquén, Río Negro, La Pampa y el norte de Chubut. Cada
polígono trae nombre de unidad, descripción litológica, génesis, edad y
morfología. Las reglas están en `targets.py` y se pueden leer completas: son
palabras clave sobre texto normalizado, y un orden de decisión. Primero se
excluye lo volcánico, lo evaporítico y lo que no menciona arena; después se
separa lo eólico cuaternario, lo fluvial cuaternario y las areniscas antiguas.

In [2]:
units = load_layer(DATA_RAW / "segemar_unidades.parquet")
print(f"{len(units):,} polígonos en la ventana, {units['nom_hoja'].nunique()} hojas 1:250.000")
for k, pat in KW.items():
    print(f"  {k:12s} {pat}")

21,546 polígonos en la ventana, 32 hojas 1:250.000
  arena        \barena[s]?\b|\barenoso[s]?\b|\barenosa[s]?\b
  arenisca     \barenisca[s]?\b
  eolico       \beolic[oa]s?\b|\bmedano[s]?\b|\bduna[s]?\b
  loess        \bloess\b|\bloessic[oa]s?\b|\bloessoide[s]?\b
  fluvial      \bfluvial(es)?\b|\baluvial(es)?\b|\bcauce[s]?\b|\bterraza[s]?\b|\bplanicie\b|\bfluvio
  cuarzo       \bcuarz(o|osa|osas|oso|osos|itica|iticas)\b|\bsilice[ao]s?\b
  finos        \blimo[s]?\b|\barcilla[s]?\b|\bfangolita[s]?\b|\bpelita[s]?\b|\blimolita[s]?\b|\barcillita[s]?\b|\blutita[s]?\b
  grava        \bgrava[s]?\b|\bconglomerado[s]?\b|\brodado[s]?\b|\bgravoso\b
  volcanico    \bvolcan|\btoba[s]?\b|\bbasalt|\bignimbrita|\blava[s]?\b|\bpiroclast|\bandesit|\briolit
  cuaternario  \bholoceno\b|\bpleistoceno\b|\bcuaternario\b|\breciente\b|\bactual(es)?\b
  evaporita    \byeso\b|\bhalita\b|\bevaporit
  carbonato    \bcaliza[s]?\b|\bcalcareo|\bcarbonat


In [3]:
t0 = time.perf_counter()
units = classify_frame(units)
units["area_km2"] = units.geometry.to_crs("EPSG:5344").area / 1e6
print(f"clasificados en {time.perf_counter() - t0:.0f} s")
resumen = units.groupby("clase").agg(poligonos=("clase", "size"), km2=("area_km2", "sum")).round(0)
resumen.loc[["arena eólica cuaternaria", "arena fluvial cuaternaria", "arenisca antigua", "otra con arena", "sin interés"]]

clasificados en 30 s


,poligonos,km2
clase,,
arena eólica cuaternaria,436,68819.0
arena fluvial cuaternaria,3142,52990.0
arenisca antigua,2405,22911.0
otra con arena,4359,98147.0
sin interés,11204,161392.0


In [4]:
for c in ("arena eólica cuaternaria", "arena fluvial cuaternaria", "arenisca antigua"):
    print(f"\n{c}: unidades con más superficie")
    print(units[units.clase == c].groupby("nombre").agg(poligonos=("nombre", "size"), km2=("area_km2", "sum")).sort_values("km2", ascending=False).head(6).round(0).to_string())
class_bar(units.assign(base=1), FIGURES / "clases.png");


arena eólica cuaternaria: unidades con más superficie


                                             poligonos     km2
nombre                                                        
Formación Meaucó facies fina                        11  8535.0
Formacion Meaucó. Facies Duna                        6  8395.0
Formación Meaucó                                    14  8041.0
Depósitos eólicos                                   79  7434.0
Formación Meaucó. Facies gruesa                      6  6602.0
Calcrete semicubierto por depósitos eólicos         24  6107.0

arena fluvial cuaternaria: unidades con más superficie
                                                         poligonos     km2
nombre                                                                    
Depósitos aluviales y coluviales                                78  6191.0
Depositos de la antigua planicie aluvial disectada              28  3961.0
Depósitos aluviales                                            270  3641.0
Depósitos Aluviales y Coluviales Indiferenciados               13

                                                          poligonos     km2
nombre                                                                     
Formación Cerro Azul                                            329  2924.0
Formación Bayo Mesa                                              10  1459.0
Subgrupo Río Neuquén (Grupo Neuquén)                             44  1315.0
Grupo Neuquén indiferenciado y/o cubierto                        20  1303.0
Formación Candeleros (Subgrupo Rio Limay, Grupo Neuquén)         20  1028.0
Formación Huincul                                               106   843.0


La clase eólica está dominada por la Formación Meaucó, el manto de arena que
cubre buena parte de La Pampa, y por los médanos holocenos de las hojas
Santa Isabel, Puelches y General Roca. Las areniscas antiguas son el Grupo
Neuquén y la Formación Cerro Azul: hay que molerlas, y por eso pesan menos.

## 2. Depósitos conocidos, rutas y consumo

Tres capas más. Los depósitos de arena, sílice y cuarzo que SEGEMAR tiene
inventariados: en la ventana son pocos y son, con nombre y apellido, las
canteras de Río Negro y de Dolavon. Las rutas nacionales, para medir cuán
lejos de un camino queda cada polígono. Y el consumo: toneladas de arena
bombeadas por área desde 2021 según el registro de fractura, con las
coordenadas de los pozos del registro de producción. El centro de gravedad de
esa demanda es el punto desde el que se miden las distancias.

In [5]:
dep = load_layer(DATA_RAW / "segemar_depositos_industriales.parquet")
txt = dep["commodity"].map(normalize) + " " + dep["nombre"].map(normalize) + " " + dep["modelo"].map(normalize)
dep_sand = dep[txt.str.contains(r"arena|silic|cuarz", regex=True)].copy()
win = box(*BBOX)
dep_win = dep_sand[dep_sand.geometry.within(win)].copy()
print(f"depósitos de arena, sílice o cuarzo: {len(dep_sand)} en el país, {len(dep_win)} en la ventana")
dep_win[["nombre", "commodity", "modelo"]].reset_index(drop=True)

depósitos de arena, sílice o cuarzo: 459 en el país, 18 en la ventana


,nombre,commodity,modelo
0,Arenas de Cercanía S.A.,arena,12g Aridos(Arenas-Gravas)
1,Flint Sand,arena silícea,12g Aridos(Arenas-Gravas)
2,El Motivo,arena silícea,12g Aridos(Arenas-Gravas)
3,CIMSA,arena,12g Aridos(Arenas-Gravas)
4,Aluvional,arena,12g Aridos(Arenas-Gravas)
5,Uan,arena,12g Aridos(Arenas-Gravas)
6,Bauti,arena silícea,12g Aridos(Arenas-Gravas)
7,Vale,arena silícea,12g Aridos(Arenas-Gravas)
8,CIMSA RN1,arena silícea,12g Aridos(Arenas-Gravas)
9,Franca,arena silícea,12g Aridos(Arenas-Gravas)


In [6]:
roads = gpd.read_file(DATA_RAW / "tmda_rutas_nacionales.geojson")
roads = roads[roads.geometry.intersects(win)].copy()
roads["geometry"] = roads.geometry.intersection(win).simplify(0.001, preserve_topology=True)
print(f"tramos de ruta nacional en la ventana: {len(roads)}")

frac = read_fracture(DATA_RAW / "fractura_adjunto_iv.csv")
coords = read_well_coords(DATA_RAW / "produccion_no_convencional.csv")
demand = demand_by_area(frac, coords, since=2021)
centro = demand_centroid(demand)
print(f"{len(demand)} áreas con arena bombeada desde 2021, {demand.sand_t.sum() / 1e6:.1f} millones de toneladas")
print(f"centro de la demanda: {centro.iloc[0].x:.2f}, {centro.iloc[0].y:.2f}")
demand[["area", "sand_t", "wells"]].head(8).assign(sand_t=lambda d: (d.sand_t / 1e3).round(0)).rename(columns={"sand_t": "kt"})

tramos de ruta nacional en la ventana: 154


55 áreas con arena bombeada desde 2021, 19.3 millones de toneladas
centro de la demanda: -68.76, -38.29


,area,kt,wells
36,LOMA CAMPANA,2602.0,233
26,LA AMARGA CHICA,2542.0,250
15,BANDURRIA SUR,2129.0,203
11,BAJADA DEL PALO OESTE,1651.0,140
25,FORTIN DE PIEDRA,1024.0,90
29,LA CALERA,966.0,90
27,LA ANGOSTURA SUR I,847.0,65
3,AGUADA DEL CHAÑAR,675.0,60


## 3. El puntaje

Un puntaje de 0 a 100 por polígono. La base la pone la clase: 1 para arena
eólica cuaternaria, 0,65 fluvial, 0,45 arenisca antigua, 0,2 otra. Suma 0,15
si la descripción menciona cuarzo o sílice, resta 0,15 si menciona limos o
arcillas y 0,10 si menciona gravas. Después la geografía: un factor que vale 1
hasta 100 km del centro de la demanda y cae a 0 a los 500 km, otro que vale 1 a
menos de 5 km de una ruta nacional y cae a 0 a los 50 km, y un bono de 15
puntos si hay un depósito de arena conocido a menos de 25 km. Todo está en
`WEIGHTS` y se puede cambiar.

In [7]:
WEIGHTS

{'clase': {'arena eólica cuaternaria': 1.0,
  'arena fluvial cuaternaria': 0.65,
  'arenisca antigua': 0.45,
  'otra con arena': 0.2,
  'sin interés': 0.0},
 'cuarzo': 0.15,
 'finos': -0.15,
 'grava': -0.1,
 'consumo_km': (100.0, 500.0),
 'ruta_km': (5.0, 50.0),
 'deposito_km': 25.0,
 'bono_deposito': 15.0}

In [8]:
t0 = time.perf_counter()
scored = prospectivity(units, roads=roads, deposits=dep_win, demand=centro)
print(f"puntaje calculado en {time.perf_counter() - t0:.0f} s")
con_base = scored[scored.base > 0]
print(f"{len(con_base):,} polígonos con algún interés; puntaje mediano {con_base.score.median():.0f}, percentil 90: {con_base.score.quantile(.9):.0f}")
scored.to_parquet(DATA_PROC / "unidades_puntuadas.parquet")

puntaje calculado en 381 s
8,994 polígonos con algún interés; puntaje mediano 20, percentil 90: 46


In [9]:
top = top_targets(scored, n=25, min_area_km2=2.0)
top.assign(area_km2=top.area_km2.round(0), d_consumo_km=top.d_consumo_km.round(0), d_ruta_km=top.d_ruta_km.round(0), d_deposito_km=top.d_deposito_km.round(0))

,nombre,nom_hoja,clase,descrip_litologica,edad_inf,area_km2,d_consumo_km,d_ruta_km,d_deposito_km,score
0,Depósito eolicos que forman medanos,GENERAL ROCA,arena eólica cuaternaria,Arenas medianas y finas,Holoceno,71.0,102.0,13.0,15.0,100.0
1,Depósitos eólicos,LONCOPUE,arena eólica cuaternaria,Arenas medianas a finas,Holoceno,4.0,160.0,7.0,188.0,91.1
2,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,39.0,194.0,1.0,141.0,88.2
3,Depósitos eólicos,LONCOPUE,arena eólica cuaternaria,Arenas medianas a finas,Holoceno,13.0,156.0,13.0,181.0,88.1
4,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,2.0,192.0,6.0,143.0,87.9
5,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,42.0,204.0,3.0,148.0,87.0
6,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,8.0,209.0,1.0,154.0,86.4
7,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,13.0,202.0,10.0,152.0,84.5
8,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,56.0,213.0,8.0,153.0,84.0
9,Depósitos eólicos,SANTA ISABEL,arena eólica cuaternaria,Arenas finas a medianas,Holoceno,9.0,194.0,14.0,149.0,83.0


## 4. Control contra lo conocido

Sin laboratorio público no hay verdad de campo. Lo que hay son lugares donde la
industria habló: Senillosa y el corredor Cutral Có–Zapala que Cormine dio por
favorables en septiembre de 2026, el Bajo de Añelo en estudio, la cantera de
Vista en Bajada del Palo, las canteras de Allen que las operadoras dejaron por
rotura y arcillas, y Dolavon. Para cada uno, el mejor puntaje a 15 km y en qué
percentil queda entre todos los polígonos con arena.

In [10]:
chk = score_at(scored, known_points(), radius_km=15)
display(chk)
summary(chk)

,lugar,estado,mejor_score,clase,unidad,percentil,mapa_mas_cercano_km
0,"Senillosa, Neuquén",favorable,70.0,arena fluvial cuaternaria,Río Limay. Depósitos fluviales antiguos,99.1,0.0
1,"Cutral Có, Neuquén",favorable,NaN,sin cobertura del mapa 1:250.000,NaN,NaN,19.9
2,"Zapala, Neuquén",favorable,NaN,sin cobertura del mapa 1:250.000,NaN,NaN,37.3
3,"Bajo de Añelo, Neuquén",en estudio,40.7,arena fluvial cuaternaria,Ríos Neuquén y Negro. Depósitos fluviales antiguos (Nive...,85.8,0.0
4,"Bajada del Palo, Neuquén",explotada,59.5,arena eólica cuaternaria,Depósitos eólicos,97.7,0.0
5,"Allen, Río Negro","explotada, calidad cuestionada",100.0,arena eólica cuaternaria,Depósito eolicos que forman medanos,100.0,0.0
6,"Dolavon, Chubut",explotada,42.5,arena fluvial cuaternaria,Depósitos fluviales de V nivel de terraza,87.7,0.0


{'favorables_mediana_percentil': 99.1,
 'cuestionados_mediana_percentil': 100.0,
 'n_favorables': 3,
 'n_cuestionados': 1}

Dos lecturas. La primera: donde hay arena, el puntaje la ve. Senillosa, Bajada
del Palo y Allen quedan arriba del percentil 95. La segunda es la que importa:
Allen, la arena que la industria descartó por calidad, sale con el puntaje
máximo. El mapa geológico dice dónde hay médanos, no si sus granos aguantan la
presión. Es exactamente el límite que anunciamos, y ahora está medido.

Y un tercer dato que no esperábamos: Cutral Có y Zapala no tienen ningún
polígono a 15 km. La cartografía 1:250.000 que publica el SIGAM tiene un hueco
entre Zapala y Piedra del Águila, y el corredor que Neuquén explora cae adentro.

In [11]:
sin = chk[chk["clase"].str.startswith("sin cobertura")]
sin[["lugar", "estado", "mapa_mas_cercano_km"]]

,lugar,estado,mapa_mas_cercano_km
1,"Cutral Có, Neuquén",favorable,19.9
2,"Zapala, Neuquén",favorable,37.3


## 5. La ruta de la arena

La arena que hoy se bombea sale de Ibicuy y llega a Añelo en camión. Los
informes de prensa de septiembre de 2026 describen el recorrido tramo por
tramo, y con esos puntos de paso un ruteador abierto sobre OpenStreetMap
devuelve la traza por caminos reales. Sirve para dos cosas: medir el viaje, y
ver qué blancos del mapa quedan al costado del camino que los camiones ya
recorren.

In [12]:
route = sand_route(cache=DATA_RAW / "ruta_arena_osrm.json")
wpts = waypoints_frame()
r = route.iloc[0]
print(f"{r['nombre']}: {r['km']:,.0f} km por {r['fuente']}, {r['horas_osrm']} h de manejo continuo según el ruteador")
scored["d_ruta_arena_km"] = distance_to_route(scored.geometry.representative_point(), route)
al_costado = scored[(scored.base > 0) & (scored.d_ruta_arena_km <= 15) & (scored.score >= 40)].copy()
al_costado["area_km2"] = al_costado.geometry.to_crs("EPSG:5344").area / 1e6
print(f"blancos con puntaje ≥ 40 a menos de 15 km de la ruta: {len(al_costado)} polígonos, {al_costado.area_km2.sum():,.0f} km²")
al_costado.groupby(["nom_hoja", "clase"]).agg(poligonos=("clase", "size"), km2=("area_km2", "sum"), score_max=("score", "max")).round(0).sort_values("km2", ascending=False).head(10)

ruta de la arena: Ibicuy a Añelo: 1,461 km por osrm, 20.2 h de manejo continuo según el ruteador


blancos con puntaje ≥ 40 a menos de 15 km de la ruta: 153 polígonos, 13,517 km²


poligonos     km2  score_max
nom_hoja       clase                                                  
GENERAL ACHA   arena eólica cuaternaria          10  7142.0       75.0
SANTA ROSA     arena eólica cuaternaria          51  3279.0       60.0
NEUQUEN        arena fluvial cuaternaria         64  1589.0       70.0
DARREGUEIRA    arena eólica cuaternaria          11   991.0       64.0
CASA DE PIEDRA arena fluvial cuaternaria          2   440.0       50.0
PUELCHES       arena fluvial cuaternaria          4    37.0       41.0
               arena eólica cuaternaria           2    27.0       80.0
NEUQUEN        arenisca antigua                   1     7.0       45.0
CASA DE PIEDRA arena eólica cuaternaria           8     5.0       84.0

## 6. El mapa

In [13]:
static_map(scored, dep_win, known_points(), FIGURES / "mapa_arena.png")
roads_all = gpd.read_file(DATA_RAW / "tmda_rutas_nacionales.geojson")
roads_all["geometry"] = roads_all.geometry.simplify(0.002, preserve_topology=True)
route_figure(route, wpts, roads_all, scored, FIGURES / "ruta_arena.png")
interactive_map(scored, dep_win, demand, known_points(), roads, min_score=30, min_area_km2=1.0, tol_deg=0.003, path=DOCS / "mapa_arena.html", route=route, waypoints=wpts)
demand.to_parquet(DATA_PROC / "demanda_por_area.parquet"); dep_win.to_parquet(DATA_PROC / "depositos_arena_ventana.parquet"); route.to_parquet(DATA_PROC / "ruta_arena.parquet")
IFrame("../docs/mapa_arena.html", width="100%", height=600)

## Límites y qué sigue

- El mapa señala dónde buscar, no si la arena sirve. Esfericidad, redondez,
  rotura y turbidez salen del laboratorio, y ningún dato público las tiene.
- La escala es 1:250.000: un médano chico no aparece, y la descripción
  litológica de cada hoja la escribió un geólogo distinto con palabras
  distintas. Las reglas son de palabras clave, y se pueden equivocar.
- Hay hojas que faltan en el SIGAM, y una de ellas es la que más importa.
- Los pesos son una opinión declarada. Cambiarlos cambia el mapa; por eso
  están en un solo lugar.

Sigue: sumar la geoquímica de SEGEMAR como proxy de pureza, incorporar los
cateos de Neuquén cuando se publiquen como datos, y un laboratorio casero de
forma de grano con microscopio USB y visión por computadora.